# Loss Functions: Building Intuition from the Ground Up

**Learning Objectives:**

By the end of this notebook, you'll have solid intuitions about:
- What loss functions are and why they're the heart of machine learning
- How different loss functions behave mathematically and geometrically
- When to use MSE, MAE, Cross-Entropy, and Hinge Loss
- The trade-offs and characteristics of each loss function
- How your choice affects training and predictions

Let's build these intuitions step by step, with lots of visualizations and hands-on experiments!

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

from aiml_notebooks import get_device, set_seed

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Set seed for reproducibility
set_seed(42)

# Get device
device = get_device()

---
## Part 1: What is a Loss Function?

### The Core Idea

Imagine you're teaching a child to throw darts at a target. After each throw, you need to tell them how far off they were. That's exactly what a **loss function** does in machine learning!

**A loss function measures how wrong your model's predictions are.**

- **Lower loss** = better predictions (closer to the target)
- **Higher loss** = worse predictions (farther from the target)

The goal of training is simple: adjust the model's parameters to make the loss as small as possible.

### Why Do We Need Different Loss Functions?

Just like different sports use different scoring systems, different ML problems need different ways to measure error:

- **Predicting house prices** (regression): How far off is our dollar amount?
- **Classifying images** (classification): How confident are we in the right class?
- **Detecting spam** (binary classification): Did we get it right or wrong?

Each scenario needs a loss function that captures what "good" means for that specific problem.

### The Learning Landscape

Think of loss as a landscape where:
- **Height** represents the loss value
- **Position** represents different model parameters
- **Goal** is to find the lowest valley (minimum loss)

Let's visualize this concept:

In [ ]:
# Create a simple 2D loss landscape
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)

# Loss function: a simple bowl shape with a minimum at (0, 0)
Z = X**2 + Y**2

fig = plt.figure(figsize=(14, 5))

# 3D view
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap=cm.coolwarm, alpha=0.8)
ax1.set_xlabel('Parameter 1')
ax1.set_ylabel('Parameter 2')
ax1.set_zlabel('Loss')
ax1.set_title('Loss Landscape (3D View)')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# Contour view (bird's eye)
ax2 = fig.add_subplot(122)
contour = ax2.contour(X, Y, Z, levels=20, cmap='coolwarm')
ax2.plot(0, 0, 'r*', markersize=20, label='Minimum (Goal!)')
ax2.set_xlabel('Parameter 1')
ax2.set_ylabel('Parameter 2')
ax2.set_title('Loss Landscape (Contour View)')
ax2.legend()
plt.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.show()

print("The red star marks the minimum loss - that's where we want to be!")
print("Training is the process of navigating this landscape to find the minimum.")

**Key Insight:** Different loss functions create different landscapes. Some are smooth and easy to navigate, others have multiple valleys or steep cliffs. Choosing the right loss function is like choosing the right map for your journey!

---
## Part 2: Regression Loss Functions

Regression problems involve predicting **continuous values** (like temperature, price, or age). Let's explore the two most popular regression loss functions.

### Mean Squared Error (MSE)

**The Idea:** Square the difference between prediction and truth, then average.

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

where:
- $y_i$ is the true value
- $\hat{y}_i$ is the predicted value
- $n$ is the number of samples

**Why square?** Squaring has special properties:
1. Makes all errors positive (no negative errors canceling positive ones)
2. Penalizes large errors much more than small errors
3. Creates smooth gradients for optimization

In [ ]:
# Let's see how MSE behaves for different errors
errors = np.linspace(-5, 5, 100)
mse_values = errors ** 2

plt.figure(figsize=(10, 6))
plt.plot(errors, mse_values, 'b-', linewidth=2, label='MSE = error²')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Error (prediction - truth)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Mean Squared Error: How It Penalizes Different Errors', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Annotate key points
plt.plot([-3, 3], [9, 9], 'ro', markersize=10)
plt.annotate('Large error\nLoss = 9', xy=(3, 9), xytext=(3.5, 15),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')
plt.plot([-1, 1], [1, 1], 'go', markersize=10)
plt.annotate('Small error\nLoss = 1', xy=(1, 1), xytext=(1.5, 5),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=10, color='green')

plt.show()

print("Notice: The curve gets steeper as error increases!")
print("An error of 3 gives loss = 9, but error of 1 gives loss = 1")
print("This means MSE REALLY punishes large errors.")

### MSE and Outliers

Let's see what happens when we have outliers (extreme values that are far from the rest):

In [ ]:
# Create data: true line with some outliers
np.random.seed(42)
n_points = 20
x_data = np.linspace(0, 10, n_points)
y_true = 2 * x_data + 1  # True relationship: y = 2x + 1

# Add noise
y_noisy = y_true + np.random.normal(0, 1, n_points)

# Add outliers
y_outliers = y_noisy.copy()
y_outliers[5] = 25  # Big outlier!
y_outliers[15] = -5  # Another outlier

# Make predictions (let's say our model predicts the true line)
y_pred = y_true

# Calculate MSE for each point
squared_errors = (y_outliers - y_pred) ** 2

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left plot: The data and predictions
axes[0].plot(x_data, y_true, 'g-', linewidth=2, label='True line (our prediction)', alpha=0.7)
axes[0].scatter(x_data, y_outliers, c='blue', s=100, alpha=0.6, edgecolors='black', label='Actual data')
axes[0].scatter(x_data[5], y_outliers[5], c='red', s=200, edgecolors='black', 
                linewidths=2, label='Outliers', zorder=5)
axes[0].scatter(x_data[15], y_outliers[15], c='red', s=200, edgecolors='black', linewidths=2, zorder=5)
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Data with Outliers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: Squared errors
colors = ['red' if i in [5, 15] else 'blue' for i in range(n_points)]
bars = axes[1].bar(range(n_points), squared_errors, color=colors, alpha=0.6, edgecolor='black')
axes[1].set_xlabel('Data Point Index')
axes[1].set_ylabel('Squared Error')
axes[1].set_title('MSE Contribution per Point')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

total_mse = np.mean(squared_errors)
outlier_contribution = (squared_errors[5] + squared_errors[15]) / np.sum(squared_errors) * 100

print(f"Total MSE: {total_mse:.2f}")
print(f"\nOutlier contribution to total loss: {outlier_contribution:.1f}%")
print(f"\nThink about it: Just 2 outliers out of {n_points} points dominate the loss!")
print("This means during training, the model will focus heavily on fitting these outliers.")

**🤔 Reflection Question:**

Is it always good that the model focuses on outliers? When might this be a problem?

### Mean Absolute Error (MAE)

**The Idea:** Take the absolute difference between prediction and truth, then average.

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

**Key Difference from MSE:** Linear penalty instead of quadratic.

Let's compare them visually:

In [ ]:
errors = np.linspace(-5, 5, 100)
mse_values = errors ** 2
mae_values = np.abs(errors)

plt.figure(figsize=(12, 6))
plt.plot(errors, mse_values, 'b-', linewidth=3, label='MSE = error²', alpha=0.7)
plt.plot(errors, mae_values, 'r-', linewidth=3, label='MAE = |error|', alpha=0.7)
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Error (prediction - truth)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('MSE vs MAE: How They Penalize Errors Differently', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.ylim(0, 20)

# Highlight the difference at large errors
error_point = 4
plt.plot([error_point, error_point], [0, error_point**2], 'b--', alpha=0.5)
plt.plot([error_point, error_point], [0, error_point], 'r--', alpha=0.5)
plt.annotate(f'At error={error_point}:\nMSE={error_point**2}, MAE={error_point}', 
            xy=(error_point, 10), fontsize=11,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.show()

print("Key Observations:")
print("1. For small errors (|error| < 1): MSE is actually smaller than MAE")
print("2. For large errors (|error| > 1): MSE grows much faster than MAE")
print("3. MAE increases linearly - it treats all errors proportionally")
print("4. MSE increases quadratically - it REALLY punishes large errors")

### MAE with Outliers

Let's see how MAE handles the same outlier data:

In [ ]:
# Using the same data from before
absolute_errors = np.abs(y_outliers - y_pred)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Squared errors (MSE)
colors = ['red' if i in [5, 15] else 'blue' for i in range(n_points)]
axes[0].bar(range(n_points), squared_errors, color=colors, alpha=0.6, edgecolor='black')
axes[0].set_xlabel('Data Point Index')
axes[0].set_ylabel('Squared Error')
axes[0].set_title('MSE Contribution per Point')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, max(squared_errors) * 1.1)

# Right: Absolute errors (MAE)
axes[1].bar(range(n_points), absolute_errors, color=colors, alpha=0.6, edgecolor='black')
axes[1].set_xlabel('Data Point Index')
axes[1].set_ylabel('Absolute Error')
axes[1].set_title('MAE Contribution per Point')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, max(squared_errors) * 1.1)  # Same scale for comparison

plt.tight_layout()
plt.show()

total_mae = np.mean(absolute_errors)
outlier_contribution_mae = (absolute_errors[5] + absolute_errors[15]) / np.sum(absolute_errors) * 100

print(f"MSE: {total_mse:.2f} | Outlier contribution: {outlier_contribution:.1f}%")
print(f"MAE: {total_mae:.2f} | Outlier contribution: {outlier_contribution_mae:.1f}%")
print(f"\n💡 Insight: With MAE, outliers have less influence on the total loss!")
print(f"This makes MAE more 'robust' to outliers.")

### MSE vs MAE: Side-by-Side Comparison

Let's create different scenarios and see how each loss function behaves:

In [ ]:
# Create three scenarios
np.random.seed(42)
x = np.linspace(0, 10, 30)
y_true = 2 * x + 1

# Scenario 1: Clean data (small noise)
y_clean = y_true + np.random.normal(0, 0.5, len(x))

# Scenario 2: Moderate noise
y_moderate = y_true + np.random.normal(0, 2, len(x))

# Scenario 3: With outliers
y_with_outliers = y_true + np.random.normal(0, 1, len(x))
y_with_outliers[8] = 30
y_with_outliers[22] = -10
y_with_outliers[15] = 25

scenarios = [
    ('Clean Data', y_clean),
    ('Moderate Noise', y_moderate),
    ('With Outliers', y_with_outliers)
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, (title, y_data) in enumerate(scenarios):
    # Top row: data
    axes[0, idx].scatter(x, y_data, alpha=0.6, s=50, edgecolors='black')
    axes[0, idx].plot(x, y_true, 'r-', linewidth=2, label='True line')
    axes[0, idx].set_title(title, fontsize=12, fontweight='bold')
    axes[0, idx].set_xlabel('X')
    axes[0, idx].set_ylabel('Y')
    axes[0, idx].legend()
    axes[0, idx].grid(True, alpha=0.3)
    
    # Bottom row: loss comparison
    mse = np.mean((y_data - y_true) ** 2)
    mae = np.mean(np.abs(y_data - y_true))
    
    losses = [mse, mae]
    labels = ['MSE', 'MAE']
    colors_bar = ['blue', 'orange']
    
    bars = axes[1, idx].bar(labels, losses, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
    axes[1, idx].set_ylabel('Loss Value')
    axes[1, idx].set_title(f'Loss Comparison\nMSE={mse:.2f}, MAE={mae:.2f}', fontsize=11)
    axes[1, idx].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, loss in zip(bars, losses):
        height = bar.get_height()
        axes[1, idx].text(bar.get_x() + bar.get_width()/2., height,
                         f'{loss:.2f}',
                         ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("1. Clean data: MSE and MAE are comparable")
print("2. Moderate noise: MSE increases faster than MAE")
print("3. With outliers: MSE shoots up dramatically, MAE stays more stable")

### When to Use MSE vs MAE?

| Aspect | MSE | MAE |
|--------|-----|-----|
| **Outlier Sensitivity** | Very sensitive | More robust |
| **Gradient** | Smooth, proportional to error | Constant (discontinuous at 0) |
| **Interpretation** | Not in original units (squared) | In original units |
| **Use Case** | Clean data, want smooth optimization | Noisy/outlier-prone data |
| **Focus** | Penalizes large errors heavily | Treats all errors equally |

**Rule of Thumb:**
- Use **MSE** when large errors are especially bad and you want to avoid them at all costs
- Use **MAE** when outliers are present or all errors should be weighted equally

---
## Part 3: Classification Loss Functions

Classification is fundamentally different from regression: instead of predicting continuous values, we're predicting **discrete categories** (classes). This requires different loss functions that work with probabilities.

### Binary Cross-Entropy Loss

**The Scenario:** Two classes (e.g., spam vs not spam, cat vs dog)

**The Idea:** Measure how different your predicted probabilities are from the true labels.

$$\text{BCE} = -\frac{1}{n}\sum_{i=1}^{n}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

where:
- $y_i \in \{0, 1\}$ is the true label
- $\hat{y}_i \in [0, 1]$ is the predicted probability

**Breaking It Down:**
- If true label is 1 (positive class): loss = $-\log(\hat{y})$
- If true label is 0 (negative class): loss = $-\log(1-\hat{y})$

Let's visualize what this means:

In [ ]:
# Predicted probabilities
probs = np.linspace(0.001, 0.999, 1000)  # Avoid log(0)

# Loss when true label is 1
loss_when_true_1 = -np.log(probs)

# Loss when true label is 0
loss_when_true_0 = -np.log(1 - probs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: True label = 1
axes[0].plot(probs, loss_when_true_1, 'b-', linewidth=3)
axes[0].set_xlabel('Predicted Probability', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Binary Cross-Entropy When True Label = 1\n(e.g., true class is "cat")', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 8)

# Annotate key points
axes[0].plot(0.9, -np.log(0.9), 'go', markersize=15)
axes[0].annotate('Confident & Correct\n(Low loss)', xy=(0.9, -np.log(0.9)), 
                xytext=(0.7, 2), arrowprops=dict(arrowstyle='->', color='green', lw=2),
                fontsize=11, color='green', fontweight='bold')

axes[0].plot(0.1, -np.log(0.1), 'ro', markersize=15)
axes[0].annotate('Confident & Wrong\n(High loss!)', xy=(0.1, -np.log(0.1)), 
                xytext=(0.3, 5), arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=11, color='red', fontweight='bold')

# Right plot: True label = 0
axes[1].plot(probs, loss_when_true_0, 'r-', linewidth=3)
axes[1].set_xlabel('Predicted Probability', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Binary Cross-Entropy When True Label = 0\n(e.g., true class is "dog")', fontsize=13)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 8)

# Annotate key points
axes[1].plot(0.1, -np.log(0.9), 'go', markersize=15)
axes[1].annotate('Confident & Correct\n(Low loss)', xy=(0.1, -np.log(0.9)), 
                xytext=(0.3, 2), arrowprops=dict(arrowstyle='->', color='green', lw=2),
                fontsize=11, color='green', fontweight='bold')

axes[1].plot(0.9, -np.log(0.1), 'ro', markersize=15)
axes[1].annotate('Confident & Wrong\n(High loss!)', xy=(0.9, -np.log(0.1)), 
                xytext=(0.7, 5), arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🎯 Key Insights:")
print("1. Loss approaches 0 when you're confident AND correct")
print("2. Loss shoots to infinity when you're confident AND wrong!")
print("3. The loss curve is asymmetric - being very wrong is severely punished")
print("4. Even being slightly right (prob > 0.5) is much better than being slightly wrong")

### Interactive Exploration: Try Different Predictions

Let's see how BCE changes with different predictions:

In [ ]:
def calculate_bce(y_true, y_pred):
    """Calculate binary cross-entropy loss"""
    epsilon = 1e-7  # Small constant to avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Example scenarios
scenarios_bce = [
    {
        'title': 'Perfect Predictions',
        'y_true': np.array([1, 0, 1, 0, 1]),
        'y_pred': np.array([0.99, 0.01, 0.99, 0.01, 0.99])
    },
    {
        'title': 'Uncertain (all 0.5)',
        'y_true': np.array([1, 0, 1, 0, 1]),
        'y_pred': np.array([0.5, 0.5, 0.5, 0.5, 0.5])
    },
    {
        'title': 'Mostly Wrong',
        'y_true': np.array([1, 0, 1, 0, 1]),
        'y_pred': np.array([0.2, 0.8, 0.3, 0.7, 0.1])
    }
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, scenario in enumerate(scenarios_bce):
    y_true = scenario['y_true']
    y_pred = scenario['y_pred']
    loss = calculate_bce(y_true, y_pred)
    
    x_pos = np.arange(len(y_true))
    width = 0.35
    
    axes[idx].bar(x_pos - width/2, y_true, width, label='True Label', alpha=0.7, color='green', edgecolor='black')
    axes[idx].bar(x_pos + width/2, y_pred, width, label='Prediction', alpha=0.7, color='blue', edgecolor='black')
    axes[idx].set_xlabel('Sample')
    axes[idx].set_ylabel('Value')
    axes[idx].set_title(f"{scenario['title']}\nBCE Loss: {loss:.3f}", fontweight='bold')
    axes[idx].set_xticks(x_pos)
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3, axis='y')
    axes[idx].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("\n💡 Notice:")
print("• Perfect predictions → Loss ≈ 0")
print("• Uncertain (0.5) → Loss = 0.693 (natural log of 2)")
print("• Mostly wrong → Loss > 1 (much higher!)")

### Categorical Cross-Entropy Loss

**The Scenario:** Multiple classes (e.g., digit recognition: 0-9, or animal species)

**The Idea:** Generalization of binary cross-entropy to multiple classes.

$$\text{CCE} = -\frac{1}{n}\sum_{i=1}^{n}\sum_{c=1}^{C}y_{i,c}\log(\hat{y}_{i,c})$$

where:
- $C$ is the number of classes
- $y_{i,c}$ is 1 if sample $i$ belongs to class $c$, else 0 (one-hot encoded)
- $\hat{y}_{i,c}$ is the predicted probability for class $c$

**Connection to Softmax:**
Models typically output raw scores (logits), then apply softmax to get probabilities:

$$\text{softmax}(z_c) = \frac{e^{z_c}}{\sum_{j=1}^{C}e^{z_j}}$$

### Visualizing Multi-Class Classification

Let's work with a 3-class problem:

In [ ]:
# Example: classifying an image as cat, dog, or bird
classes = ['Cat', 'Dog', 'Bird']

# Scenario 1: Confident and correct (true class = Cat)
pred1 = np.array([0.9, 0.05, 0.05])
true1 = np.array([1, 0, 0])

# Scenario 2: Uncertain
pred2 = np.array([0.4, 0.35, 0.25])
true2 = np.array([1, 0, 0])

# Scenario 3: Confident but wrong
pred3 = np.array([0.1, 0.1, 0.8])
true3 = np.array([1, 0, 0])

def cross_entropy(y_true, y_pred):
    epsilon = 1e-7
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred))

scenarios_ce = [
    ('Confident & Correct', pred1, true1),
    ('Uncertain', pred2, true2),
    ('Confident & Wrong', pred3, true3)
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (title, pred, true) in enumerate(scenarios_ce):
    loss = cross_entropy(true, pred)
    
    x = np.arange(len(classes))
    width = 0.35
    
    bars1 = axes[idx].bar(x - width/2, true, width, label='True (one-hot)', 
                          alpha=0.7, color='green', edgecolor='black', linewidth=2)
    bars2 = axes[idx].bar(x + width/2, pred, width, label='Predicted Prob', 
                          alpha=0.7, color='blue', edgecolor='black', linewidth=2)
    
    axes[idx].set_ylabel('Probability', fontsize=11)
    axes[idx].set_title(f'{title}\nCross-Entropy Loss: {loss:.3f}', fontsize=12, fontweight='bold')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(classes)
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3, axis='y')
    axes[idx].set_ylim(0, 1.1)
    
    # Highlight the true class
    true_idx = np.argmax(true)
    axes[idx].axvline(x=true_idx, color='green', linestyle='--', alpha=0.3, linewidth=3)

plt.tight_layout()
plt.show()

print("\n🎯 Understanding the Loss:")
print(f"1. Confident & Correct: Loss = {cross_entropy(true1, pred1):.3f} ✓")
print(f"2. Uncertain: Loss = {cross_entropy(true2, pred2):.3f} (not terrible)")
print(f"3. Confident & Wrong: Loss = {cross_entropy(true3, pred3):.3f} ✗ (very bad!)")
print("\nThe loss only cares about the probability assigned to the TRUE class.")

### Hinge Loss (Support Vector Machine Loss)

**The Scenario:** Binary classification, but with a different philosophy

**The Idea:** Instead of probabilities, we work with raw scores. We want:
- Positive samples to have score > +1
- Negative samples to have score < -1
- A "margin" of safety between classes

$$\text{Hinge Loss} = \frac{1}{n}\sum_{i=1}^{n}\max(0, 1 - y_i \cdot f(x_i))$$

where:
- $y_i \in \{-1, +1\}$ is the true label (note: different encoding!)
- $f(x_i)$ is the raw score (not probability)

**Key Property:** Loss is 0 when the prediction is correct with a margin of at least 1.

In [ ]:
# Let's visualize hinge loss
scores = np.linspace(-3, 3, 1000)

# For a positive sample (y = +1)
hinge_loss_pos = np.maximum(0, 1 - scores)

# For a negative sample (y = -1)
hinge_loss_neg = np.maximum(0, 1 + scores)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Positive sample
axes[0].plot(scores, hinge_loss_pos, 'b-', linewidth=3)
axes[0].axvline(x=1, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Margin boundary')
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].fill_between(scores, 0, hinge_loss_pos, where=(scores < 1), alpha=0.3, color='red', label='Loss region')
axes[0].fill_between(scores, 0, 0.1, where=(scores >= 1), alpha=0.3, color='green', label='Safe zone (loss = 0)')
axes[0].set_xlabel('Model Score', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Hinge Loss for Positive Sample (y = +1)\nWant score > +1', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)
axes[0].set_ylim(-0.1, 3)

# Annotate key points
axes[0].plot(2, 0, 'go', markersize=15)
axes[0].annotate('Safe zone\n(beyond margin)', xy=(2, 0), xytext=(2.2, 1),
                arrowprops=dict(arrowstyle='->', color='green', lw=2),
                fontsize=11, color='green', fontweight='bold')

axes[0].plot(-1, 2, 'ro', markersize=15)
axes[0].annotate('Wrong side\n(high loss)', xy=(-1, 2), xytext=(-2, 2.5),
                arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=11, color='red', fontweight='bold')

# Right: Negative sample
axes[1].plot(scores, hinge_loss_neg, 'r-', linewidth=3)
axes[1].axvline(x=-1, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Margin boundary')
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].fill_between(scores, 0, hinge_loss_neg, where=(scores > -1), alpha=0.3, color='red', label='Loss region')
axes[1].fill_between(scores, 0, 0.1, where=(scores <= -1), alpha=0.3, color='green', label='Safe zone (loss = 0)')
axes[1].set_xlabel('Model Score', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Hinge Loss for Negative Sample (y = -1)\nWant score < -1', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)
axes[1].set_ylim(-0.1, 3)

# Annotate key points
axes[1].plot(-2, 0, 'go', markersize=15)
axes[1].annotate('Safe zone\n(beyond margin)', xy=(-2, 0), xytext=(-2.5, 1),
                arrowprops=dict(arrowstyle='->', color='green', lw=2),
                fontsize=11, color='green', fontweight='bold')

axes[1].plot(1, 2, 'ro', markersize=15)
axes[1].annotate('Wrong side\n(high loss)', xy=(1, 2), xytext=(1.5, 2.5),
                arrowprops=dict(arrowstyle='->', color='red', lw=2),
                fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🎯 Key Insights about Hinge Loss:")
print("1. Loss is ZERO when you're on the correct side with margin ≥ 1")
print("2. Loss increases linearly as you move toward (or past) the decision boundary")
print("3. Unlike cross-entropy, hinge loss stops caring once you're 'confident enough'")
print("4. This creates a 'margin' - a safety zone between classes")

### Cross-Entropy vs Hinge Loss

Let's compare these two losses directly:

In [ ]:
# For fair comparison, let's look at a positive sample (y = +1)
scores = np.linspace(-3, 3, 1000)

# Hinge loss (y = +1)
hinge = np.maximum(0, 1 - scores)

# For cross-entropy, convert scores to probabilities with sigmoid
probs = 1 / (1 + np.exp(-scores))  # sigmoid
cross_ent = -np.log(probs + 1e-7)  # y = 1, so BCE = -log(p)

plt.figure(figsize=(12, 7))
plt.plot(scores, hinge, 'b-', linewidth=3, label='Hinge Loss', alpha=0.8)
plt.plot(scores, cross_ent, 'r-', linewidth=3, label='Cross-Entropy Loss', alpha=0.8)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5, linewidth=2, label='Decision boundary (score=0)')
plt.axvline(x=1, color='green', linestyle='--', alpha=0.5, linewidth=2, label='Hinge margin (score=1)')
plt.xlabel('Model Score', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Cross-Entropy vs Hinge Loss (for positive sample, y=+1)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.ylim(0, 5)
plt.xlim(-3, 3)

# Annotate differences
plt.annotate('Hinge loss stops\npenalizing here', xy=(1, 0), xytext=(1.5, 1.5),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2),
            fontsize=11, color='blue', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.annotate('Cross-entropy keeps\nimproving even\nwhen very confident', xy=(2.5, -np.log(1/(1+np.exp(-2.5)))), 
            xytext=(1.8, 0.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, color='red', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

plt.show()

print("\n📊 Comparing the Philosophies:")
print("\nHinge Loss (SVM):")
print("  • Goal: Create a MARGIN of separation")
print("  • Once you're confident enough (beyond margin), loss = 0")
print("  • Doesn't care about exact probabilities")
print("  • Linear penalty")
print("\nCross-Entropy:")
print("  • Goal: Get probabilities as close to 0/1 as possible")
print("  • Always tries to improve, never satisfied")
print("  • Cares deeply about probability calibration")
print("  • Exponential penalty (severe for very wrong predictions)")

---
## Part 4: Training with Different Losses

Let's see how different loss functions affect actual training! We'll create a simple toy dataset and train models with different losses.

### Toy Dataset: Regression with Outliers

In [ ]:
# Generate synthetic data: y = 3x + 2 + noise
torch.manual_seed(42)
n_samples = 100
X_train = torch.linspace(0, 10, n_samples).unsqueeze(1)
y_train = 3 * X_train + 2 + torch.randn(n_samples, 1) * 2

# Add outliers
outlier_indices = [20, 45, 70, 85]
for idx in outlier_indices:
    y_train[idx] += torch.randn(1) * 20  # Big noise!

# Move to device
X_train = X_train.to(device)
y_train = y_train.to(device)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_train.cpu(), y_train.cpu(), alpha=0.6, s=50, edgecolors='black')
plt.scatter(X_train[outlier_indices].cpu(), y_train[outlier_indices].cpu(), 
           color='red', s=200, edgecolors='black', linewidths=2, label='Outliers', zorder=5)
plt.xlabel('X', fontsize=12)
plt.ylabel('Y', fontsize=12)
plt.title('Training Data with Outliers', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Dataset: {n_samples} points with {len(outlier_indices)} outliers")

### Train Simple Linear Models with MSE and MAE

In [ ]:
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)

def train_model(loss_fn, loss_name, epochs=1000, lr=0.01):
    model = LinearModel().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    
    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_train)
        loss = loss_fn(predictions, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return model, losses

# Train with MSE
print("Training with MSE...")
model_mse, losses_mse = train_model(nn.MSELoss(), 'MSE', epochs=500)

# Train with MAE
print("Training with MAE...")
model_mae, losses_mae = train_model(nn.L1Loss(), 'MAE', epochs=500)

print("\nTraining complete!")

### Compare the Learned Models

In [ ]:
# Get predictions
with torch.no_grad():
    y_pred_mse = model_mse(X_train)
    y_pred_mae = model_mae(X_train)

# True underlying line (without noise)
y_true_line = 3 * X_train + 2

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Fitted lines
axes[0].scatter(X_train.cpu(), y_train.cpu(), alpha=0.4, s=30, color='gray', label='Data')
axes[0].scatter(X_train[outlier_indices].cpu(), y_train[outlier_indices].cpu(), 
               color='red', s=150, edgecolors='black', linewidths=2, label='Outliers', zorder=5)
axes[0].plot(X_train.cpu(), y_true_line.cpu(), 'g--', linewidth=2, label='True line', alpha=0.7)
axes[0].plot(X_train.cpu(), y_pred_mse.cpu(), 'b-', linewidth=3, label='MSE model', alpha=0.8)
axes[0].plot(X_train.cpu(), y_pred_mae.cpu(), 'orange', linewidth=3, label='MAE model', alpha=0.8)
axes[0].set_xlabel('X', fontsize=12)
axes[0].set_ylabel('Y', fontsize=12)
axes[0].set_title('Fitted Models: MSE vs MAE', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Right: Training curves
axes[1].plot(losses_mse, 'b-', linewidth=2, label='MSE', alpha=0.7)
axes[1].plot(losses_mae, 'orange', linewidth=2, label='MAE', alpha=0.7)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print learned parameters
mse_weight = model_mse.linear.weight.item()
mse_bias = model_mse.linear.bias.item()
mae_weight = model_mae.linear.weight.item()
mae_bias = model_mae.linear.bias.item()

print("\n📊 Learned Parameters:")
print(f"\nTrue relationship: y = 3.0 * x + 2.0")
print(f"\nMSE Model: y = {mse_weight:.2f} * x + {mse_bias:.2f}")
print(f"MAE Model: y = {mae_weight:.2f} * x + {mae_bias:.2f}")
print(f"\n💡 Notice: The MAE model is typically closer to the true line!")
print(f"The MSE model gets 'pulled' more by the outliers.")

### Toy Dataset: Binary Classification

In [ ]:
# Generate 2D binary classification data
torch.manual_seed(42)
n = 200

# Class 0 (centered at (-2, -2))
X0 = torch.randn(n, 2) * 0.8 + torch.tensor([-2.0, -2.0])
y0 = torch.zeros(n, 1)

# Class 1 (centered at (2, 2))
X1 = torch.randn(n, 2) * 0.8 + torch.tensor([2.0, 2.0])
y1 = torch.ones(n, 1)

# Combine
X_class = torch.cat([X0, X1], dim=0).to(device)
y_class = torch.cat([y0, y1], dim=0).to(device)

# Shuffle
perm = torch.randperm(X_class.size(0))
X_class = X_class[perm]
y_class = y_class[perm]

# Visualize
plt.figure(figsize=(8, 8))
plt.scatter(X_class[y_class.squeeze()==0, 0].cpu(), 
           X_class[y_class.squeeze()==0, 1].cpu(), 
           c='blue', s=50, alpha=0.6, edgecolors='black', label='Class 0')
plt.scatter(X_class[y_class.squeeze()==1, 0].cpu(), 
           X_class[y_class.squeeze()==1, 1].cpu(), 
           c='red', s=50, alpha=0.6, edgecolors='black', label='Class 1')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Binary Classification Dataset', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"Dataset: {X_class.shape[0]} points, 2 classes")

### Train with Binary Cross-Entropy and Hinge Loss

In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 10)
        self.fc2 = nn.Linear(10, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)  # Raw logits

def train_classifier(model, loss_fn, X, y, epochs=500, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")
    
    return losses

# Model with BCE
print("Training with Binary Cross-Entropy...\n")
model_bce = BinaryClassifier().to(device)
bce_loss = nn.BCEWithLogitsLoss()  # Combines sigmoid + BCE for numerical stability
losses_bce = train_classifier(model_bce, bce_loss, X_class, y_class, epochs=500)

# Model with Hinge Loss
print("\nTraining with Hinge Loss...\n")
model_hinge = BinaryClassifier().to(device)
# Convert y from {0,1} to {-1,+1} for hinge loss
y_hinge = 2 * y_class - 1  # 0 -> -1, 1 -> +1
hinge_loss = lambda outputs, targets: torch.mean(torch.clamp(1 - targets * outputs, min=0))
losses_hinge = train_classifier(model_hinge, hinge_loss, X_class, y_hinge, epochs=500)

print("\nTraining complete!")

### Visualize Decision Boundaries

In [ ]:
def plot_decision_boundary(model, X, y, title, ax):
    # Create mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min.cpu(), x_max.cpu(), 200),
                        np.linspace(y_min.cpu(), y_max.cpu(), 200))
    
    # Predict on mesh
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32).to(device)
    with torch.no_grad():
        Z = model(grid)
        Z = torch.sigmoid(Z)  # Convert to probabilities
    Z = Z.cpu().numpy().reshape(xx.shape)
    
    # Plot
    contour = ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=3)
    
    # Plot data
    y_np = y.cpu().numpy().squeeze()
    # Convert y_hinge back to {0, 1} for plotting
    if y_np.min() < 0:
        y_np = (y_np + 1) / 2
    
    ax.scatter(X[y_np==0, 0].cpu(), X[y_np==0, 1].cpu(), 
              c='blue', s=30, edgecolors='black', alpha=0.7, label='Class 0')
    ax.scatter(X[y_np==1, 0].cpu(), X[y_np==1, 1].cpu(), 
              c='red', s=30, edgecolors='black', alpha=0.7, label='Class 1')
    
    ax.set_xlabel('Feature 1', fontsize=11)
    ax.set_ylabel('Feature 2', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    return contour

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Top row: Decision boundaries
plot_decision_boundary(model_bce, X_class, y_class, 'Binary Cross-Entropy\nDecision Boundary', axes[0, 0])
plot_decision_boundary(model_hinge, X_class, y_hinge, 'Hinge Loss\nDecision Boundary', axes[0, 1])

# Bottom left: Training curves
axes[1, 0].plot(losses_bce, 'b-', linewidth=2, label='BCE', alpha=0.7)
axes[1, 0].plot(losses_hinge, 'orange', linewidth=2, label='Hinge', alpha=0.7)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Loss', fontsize=11)
axes[1, 0].set_title('Training Loss Curves', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)

# Bottom right: Accuracy comparison
with torch.no_grad():
    # BCE predictions
    pred_bce = (torch.sigmoid(model_bce(X_class)) > 0.5).float()
    acc_bce = (pred_bce == y_class).float().mean().item()
    
    # Hinge predictions
    pred_hinge = (model_hinge(X_class) > 0).float()
    y_class_01 = (y_hinge + 1) / 2  # Convert back to {0, 1}
    acc_hinge = (pred_hinge == y_class_01).float().mean().item()

accuracies = [acc_bce * 100, acc_hinge * 100]
labels_acc = ['BCE', 'Hinge']
colors_acc = ['blue', 'orange']

bars = axes[1, 1].bar(labels_acc, accuracies, color=colors_acc, alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1, 1].set_title('Classification Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim(0, 105)
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{acc:.1f}%',
                   ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print(f"\n📊 Results:")
print(f"BCE Accuracy: {acc_bce*100:.1f}%")
print(f"Hinge Accuracy: {acc_hinge*100:.1f}%")
print(f"\nBoth perform similarly, but notice the different decision boundaries!")

---
## Part 5: Practical Guidance

### Decision Tree: Which Loss Function to Use?

```
┌─ What are you predicting?
│
├─ CONTINUOUS VALUES (Regression)
│  │
│  ├─ Do you have outliers?
│  │  ├─ YES → Use MAE (robust to outliers)
│  │  └─ NO  → Use MSE (smooth gradients, penalizes large errors)
│  │
│  └─ Special cases:
│     ├─ Need interpretability in original units? → MAE
│     └─ Large errors are especially bad? → MSE
│
└─ DISCRETE CLASSES (Classification)
   │
   ├─ How many classes?
   │  │
   │  ├─ TWO (Binary)
   │  │  ├─ Need probability estimates? → Binary Cross-Entropy
   │  │  ├─ Need maximum margin? → Hinge Loss (SVM)
   │  │  └─ Multi-label (multiple classes can be true)? → BCE per label
   │  │
   │  └─ MULTIPLE (Multi-class)
   │     └─ Use Categorical Cross-Entropy (with Softmax)
   │
   └─ Special considerations:
      ├─ Imbalanced classes? → Weighted Cross-Entropy or Focal Loss
      ├─ Need calibrated probabilities? → Cross-Entropy
      └─ SVM-style geometric margin? → Hinge Loss
```

### Common Pitfalls and Solutions

#### Pitfall 1: Using the Wrong Labels
- **MSE/MAE**: Labels are continuous values
- **Binary Cross-Entropy**: Labels are in {0, 1}
- **Hinge Loss**: Labels are in {-1, +1}
- **Categorical Cross-Entropy**: Labels are class indices (0, 1, 2, ...) or one-hot encoded

#### Pitfall 2: Forgetting the Activation Function
- **MSE/MAE**: No activation (raw output)
- **Binary Cross-Entropy**: Use sigmoid (or BCEWithLogitsLoss)
- **Hinge Loss**: No activation (raw scores)
- **Categorical Cross-Entropy**: Use softmax (or CrossEntropyLoss)

#### Pitfall 3: Numerical Instability
- Never compute log(0) or divide by zero
- Use `BCEWithLogitsLoss` instead of BCE + sigmoid
- Use `CrossEntropyLoss` instead of softmax + log
- Add small epsilon (1e-7) when computing logs manually

#### Pitfall 4: Wrong Loss for the Task
- Don't use MSE for classification (it works but poorly!)
- Don't use Cross-Entropy for regression
- Consider your evaluation metric - if you're measuring MAE, train with MAE

### Loss Function Properties Summary

| Loss Function | Task | Output | Range | Outlier Sensitive | Gradient | When to Use |
|---------------|------|--------|-------|-------------------|----------|-------------|
| **MSE** | Regression | Continuous | [0, ∞) | Very | Smooth, ∝ error | Clean data, want to avoid large errors |
| **MAE** | Regression | Continuous | [0, ∞) | Less | Constant | Noisy/outlier-prone data |
| **BCE** | Binary Class | Probability | [0, ∞) | Very | Large for wrong | Need probabilities, binary classification |
| **CCE** | Multi-class | Probability | [0, ∞) | Very | Large for wrong | Multi-class classification |
| **Hinge** | Binary Class | Raw score | [0, ∞) | Medium | 0 or constant | SVM, want margin, don't need probabilities |

---
## Part 6: Hands-on Experiments

Now it's your turn! Try these experiments to deepen your understanding.

### Experiment 1: Effect of Outlier Strength

**Task:** Modify the outlier strength in the regression dataset and observe how MSE vs MAE models respond.

Try changing the outlier magnitude (currently `* 20`) and retrain both models. What happens to the fitted lines?

In [ ]:
# Experiment here!
# Hint: Copy the regression training code above and modify the outlier strength

outlier_strength = 10  # Try: 5, 10, 20, 50

# Generate data
torch.manual_seed(42)
X_exp = torch.linspace(0, 10, 100).unsqueeze(1).to(device)
y_exp = 3 * X_exp + 2 + torch.randn(100, 1).to(device) * 2

# Add outliers with different strength
for idx in [20, 45, 70, 85]:
    y_exp[idx] += torch.randn(1).to(device) * outlier_strength

# Train both models
model_mse_exp, _ = train_model(nn.MSELoss(), 'MSE', epochs=500)
model_mae_exp, _ = train_model(nn.L1Loss(), 'MAE', epochs=500)

# Plot
with torch.no_grad():
    plt.figure(figsize=(10, 6))
    plt.scatter(X_exp.cpu(), y_exp.cpu(), alpha=0.4, s=30, color='gray')
    plt.plot(X_exp.cpu(), (3*X_exp + 2).cpu(), 'g--', linewidth=2, label='True line')
    plt.plot(X_exp.cpu(), model_mse_exp(X_exp).cpu(), 'b-', linewidth=2, label='MSE')
    plt.plot(X_exp.cpu(), model_mae_exp(X_exp).cpu(), 'orange', linewidth=2, label='MAE')
    plt.title(f'Outlier Strength = {outlier_strength}', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print(f"\n🔬 Experiment with outlier_strength = {outlier_strength}")
print(f"Try different values and observe the difference between MSE and MAE!")

### Experiment 2: Visualize Loss Surfaces

**Task:** For a simple 2-parameter model, visualize how the loss landscape looks with different loss functions.

In [ ]:
# Simple dataset: fit y = mx + b to 10 points
torch.manual_seed(42)
x_simple = torch.linspace(0, 10, 10)
y_simple = 2 * x_simple + 3 + torch.randn(10) * 0.5

# Try different parameter values
m_range = np.linspace(0, 4, 50)
b_range = np.linspace(0, 6, 50)
M, B = np.meshgrid(m_range, b_range)

# Compute MSE for each parameter combination
MSE_surface = np.zeros_like(M)
MAE_surface = np.zeros_like(M)

for i in range(len(m_range)):
    for j in range(len(b_range)):
        m, b = M[j, i], B[j, i]
        predictions = m * x_simple + b
        MSE_surface[j, i] = torch.mean((y_simple - predictions) ** 2).item()
        MAE_surface[j, i] = torch.mean(torch.abs(y_simple - predictions)).item()

fig = plt.figure(figsize=(16, 6))

# MSE surface
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(M, B, MSE_surface, cmap=cm.viridis, alpha=0.8)
ax1.set_xlabel('Slope (m)', fontsize=11)
ax1.set_ylabel('Intercept (b)', fontsize=11)
ax1.set_zlabel('MSE Loss', fontsize=11)
ax1.set_title('MSE Loss Surface', fontsize=13, fontweight='bold')
fig.colorbar(surf1, ax=ax1, shrink=0.5)

# MAE surface
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(M, B, MAE_surface, cmap=cm.plasma, alpha=0.8)
ax2.set_xlabel('Slope (m)', fontsize=11)
ax2.set_ylabel('Intercept (b)', fontsize=11)
ax2.set_zlabel('MAE Loss', fontsize=11)
ax2.set_title('MAE Loss Surface', fontsize=13, fontweight='bold')
fig.colorbar(surf2, ax=ax2, shrink=0.5)

plt.tight_layout()
plt.show()

print("\n🗺️ Observe the loss landscapes:")
print("• MSE creates a smooth, bowl-shaped surface")
print("• MAE creates a surface with sharper features (notice the 'ridge' pattern)")
print("• Both have a clear minimum where the true parameters are!")

### Experiment 3: Your Own Dataset!

**Task:** Create your own dataset and try different loss functions.

Some ideas:
- Create non-linearly separable data for classification
- Add different types of noise to regression data
- Try multi-class classification with 3+ classes
- Experiment with imbalanced datasets

In [ ]:
# Your experiment here!
# Be creative and explore!

print("🎨 This is your canvas!")
print("Create interesting datasets and see how different losses behave.")
print("Some questions to explore:")
print("• What happens with very imbalanced classes?")
print("• How do losses behave with non-linear decision boundaries?")
print("• Can you find cases where MSE outperforms MAE despite outliers?")

---
## Summary and Key Takeaways

Congratulations! You've built strong intuitions about loss functions. Here's what you should remember:

### Core Concepts
1. **Loss functions measure how wrong predictions are** - they're the compass that guides training
2. **Different problems need different losses** - regression vs classification require fundamentally different approaches
3. **Loss functions create optimization landscapes** - smooth landscapes are easier to navigate

### Regression Losses
- **MSE**: Smooth, penalizes large errors heavily, sensitive to outliers
- **MAE**: Robust, treats all errors equally, less smooth (constant gradient)

### Classification Losses
- **Cross-Entropy**: Focuses on probability calibration, never satisfied, wants perfect confidence
- **Hinge Loss**: Focuses on margins, stops caring after achieving separation

### Practical Wisdom
- Match your loss to your evaluation metric when possible
- Consider your data characteristics (outliers, noise, class balance)
- Use stable implementations (e.g., `BCEWithLogitsLoss`, `CrossEntropyLoss`)
- Visualize! Understanding loss behavior is crucial for debugging

### Next Steps
- Explore advanced losses: Focal Loss (imbalanced data), Huber Loss (combines MSE+MAE)
- Learn about loss weighting and curriculum learning
- Study how loss functions relate to probabilistic interpretations

**Remember:** Choosing the right loss function is often as important as choosing the right architecture!

---
## Reflection Questions

Test your understanding:

1. **Why does MSE square the error instead of just using the absolute value?**

2. **In what scenario would Cross-Entropy be preferred over Hinge Loss for binary classification?**

3. **If your model is confident but wrong, which loss function punishes it most severely?**

4. **Why might you choose MAE for a regression task predicting house prices?**

5. **What's the main difference in philosophy between Cross-Entropy and Hinge Loss?**

Think about these questions and experiment with the code above to find the answers!